# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mujahid1hm/flyrank-ai-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule is: prioritize pages that already have real search demand but are underperforming or becoming stale. In plain terms, a good refresh candidate is a page with enough impressions to matter, some aging or weak ranking, and either low CTR, weak engagement, or a declining trend.

Reason codes the rule can emit:
- `stale_visible_page` — old content with enough impressions to matter
- `declining_with_demand` — declining trend while still getting visible demand
- `thin_visible_page` — too little content depth for a page that still gets traffic
- `page_one_decay_risk` — page-one ranking but aging content at risk of decay
- `low_ctr_visible_page` — visible page with weak click efficiency
- `low_engagement_visible_page` — visible page with low engagement or scroll quality
- `general_refresh_review` — fallback if none of the stronger signals fire


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path(r"c:\Users\Mujahid\Documents\flyrank-ai-")
print(f"Repo root: {repo_root}")

raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
if not raw_path.exists():
    raise FileNotFoundError(f"Starter CSV not found: {raw_path}")

processed_dir = repo_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
feature_path = processed_dir / "refresh_feature_vector.csv"
if not feature_path.exists():
    df = pd.read_csv(raw_path)
    df = df.copy()
    numeric_cols = [
        "search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d",
        "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
        "ai_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions",
        "impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
        "clicks_prev_30d", "sessions_prev_30d", "content_age_days", "age_tier_order",
        "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
        "ai_traffic_pct", "trend_pct"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        else:
            df[col] = 0
    for col in ["competition_level", "content_type", "main_intent", "provider_used", "model_used",
                "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
                "impression_tier", "position_tier", "trend_direction"]:
        if col in df.columns:
            df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
        else:
            df[col] = "unknown"

    for col in numeric_cols:
        df[col] = df[col].fillna(0)

    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
    df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
    df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
    df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
    df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
    df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
    df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
    df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)
    df.to_csv(feature_path, index=False)
    print(f"Prepared feature vector: {feature_path}")
else:
    df = pd.read_csv(feature_path)
    print(f"Loaded feature vector: {feature_path}")


def reason_codes(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and ((0 < row["engagement_rate"] < 30) or (0 < row["scroll_rate"] < 30)):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons


def normalize(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    minimum = values.min()
    maximum = values.max()
    if maximum == minimum or not np.isfinite(minimum) or not np.isfinite(maximum):
        return pd.Series(np.zeros(len(values), dtype=float), index=values.index)
    return (values - minimum) / (maximum - minimum)


def percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)

# Baseline score from the repo reference pipeline.
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

df["reason_codes"] = df.apply(lambda row: "|".join(reason_codes(row)), axis=1)
df["suggested_action_baseline"] = df["reason_codes"].apply(
    lambda codes: "expand_and_refresh" if "thin_visible_page" in str(codes).split("|")
    else "refresh_and_review_ctr" if "low_ctr_visible_page" in str(codes).split("|")
    else "refresh" if ("stale_visible_page" in str(codes).split("|") or "declining_with_demand" in str(codes).split("|"))
    else "monitor"
)
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "visibility_score", "freshness_risk_score", "position_opportunity_score",
    "depth_gap_score", "reason_codes", "suggested_action_baseline",
    "is_declining_label", "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "content_age_days", "days_since_last_update", "word_count", "trend_direction",
]

queue = df[output_columns].sort_values("baseline_rank").reset_index(drop=True)
queue_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
queue_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(queue_path, index=False)

processed_queue_path = repo_root / "data" / "processed" / "baseline_refresh_queue.csv"
queue.to_csv(processed_queue_path, index=False)

print(f"Baseline rows: {len(queue):,}")
print(f"Top score: {queue['baseline_refresh_score'].max():.3f}")
print(f"Median score: {queue['baseline_refresh_score'].median():.3f}")
print(f"Base rate (declining): {queue['is_declining_label'].mean():.3f}")
print(f"Saved work queue: {queue_path}")
print(queue.head(10)[["baseline_rank", "content_id", "baseline_refresh_score", "suggested_action_baseline", "reason_codes", "is_declining_label"]])


Repo root: c:\Users\Mujahid\Documents\flyrank-ai-
Loaded feature vector: c:\Users\Mujahid\Documents\flyrank-ai-\data\processed\refresh_feature_vector.csv
Baseline rows: 30,000
Top score: 0.941
Median score: 0.443
Base rate (declining): 0.542
Saved work queue: c:\Users\Mujahid\Documents\flyrank-ai-\work\outputs\baseline_action_score.csv
   baseline_rank            content_id  baseline_refresh_score  \
0              1  content_9532f197bbc8                0.941189   
1              2  content_4d1fe5b32dc2                0.934889   
2              3  content_07f2e7a6f38a                0.934080   
3              4  content_e5ae436f9a16                0.933606   
4              5  content_3430a8b94511                0.933559   
5              6  content_cbd93118300b                0.933263   
6              7  content_9c195417f6ef                0.932991   
7              8  content_ba2acb4ebd04                0.931623   
8              9  content_79b25654070a                0.931363   
9 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

This baseline uses a transparent weighted score rather than a fitted model: the strongest pages to review are those with visible demand, aging content, weak rankings, and lower click/engagement efficiency. The queue is then ranked by `baseline_refresh_score`, with every page carrying explicit reason codes.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 are the ones we would hand-review first. The clear winners are pages with high impressions, aging content, weak rankings, and weak click efficiency. The main failure mode is a page that looks high-volume but is not actually underperforming; those are the picks we flag as weak in the next section.


In [2]:
# Top-20 review
review = queue.head(20).copy()
review["action"] = review["suggested_action_baseline"]
review["reason_code_list"] = review["reason_codes"].str.split("|")
review["confidence_note"] = np.where(
    review["impressions_90d"] >= 2000,
    "High visibility, so this is a real candidate for action.",
    "Still worth a review, but lower-volume signal than the top tier."
)

print("Top-20 review table")
print(
    review[["baseline_rank", "content_id", "action", "reason_codes", "baseline_refresh_score", "confidence_note", "impressions_90d", "avg_position", "ctr"]]
    .round({"baseline_refresh_score": 4, "ctr": 4})
    .to_string(index=False)
)

print("\nPrecision@50 on the baseline queue:")
# Precision@50 uses the same label definition as the project pipeline.
order = queue.sort_values("baseline_refresh_score", ascending=False).head(50)
print(f"Precision@50 = {order['is_declining_label'].mean():.3f} ({order['is_declining_label'].mean() * 100:.1f}%)")
print(f"Base rate = {queue['is_declining_label'].mean():.3f} ({queue['is_declining_label'].mean() * 100:.1f}%)")


Top-20 review table
 baseline_rank           content_id                 action                                                                               reason_codes  baseline_refresh_score                                          confidence_note  impressions_90d  avg_position  ctr
             1 content_9532f197bbc8                refresh                      declining_with_demand|page_one_decay_risk|low_engagement_visible_page                  0.9412 High visibility, so this is a real candidate for action.           309192           2.0 0.87
             2 content_4d1fe5b32dc2                monitor                                            page_one_decay_risk|low_engagement_visible_page                  0.9349 High visibility, so this is a real candidate for action.            97999           2.5 0.52
             3 content_07f2e7a6f38a                monitor                                            page_one_decay_risk|low_engagement_visible_page                  0.9341 High 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The most suspicious picks are pages that rank highly only because they have a lot of impressions, but they do not show the underperformance or freshness signals that justify a refresh. Those are exactly the pages the human review should challenge. The baseline uses only observed traffic, freshness, ranking, and depth features; it does not use `trend_direction` or `trend_pct`, so the score is not leaking the label itself.


In [3]:
# Weak picks + leakage check
weak_picks = (
    queue.head(20)
    .assign(is_weak=lambda x: (x["is_declining_label"] == 0) & (x["impressions_90d"] >= 1500))
    .query("is_weak == True")
    [["baseline_rank", "content_id", "reason_codes", "suggested_action_baseline", "impressions_90d", "avg_position", "ctr", "is_declining_label"]]
)
print("Weak picks in the top 20 (likely over-prioritized because of visibility alone)")
print(weak_picks.to_string(index=False))

# Leakage guard: never use the label-derived fields as features.
feature_columns = ["impressions_90d", "avg_position", "ctr", "engagement_rate", "scroll_rate", "word_count", "content_age_days", "days_since_last_update"]
leak_guard = {"trend_direction", "trend_pct"}.isdisjoint(set(feature_columns))
print(f"\nLeakage guard: {leak_guard}")
print("Explanation: the score uses observed traffic and content hygiene features, not the label-derived or future-trend columns.")


Weak picks in the top 20 (likely over-prioritized because of visibility alone)
 baseline_rank           content_id                                                         reason_codes suggested_action_baseline  impressions_90d  avg_position  ctr  is_declining_label
             2 content_4d1fe5b32dc2                      page_one_decay_risk|low_engagement_visible_page                   monitor            97999           2.5 0.52                   0
             3 content_07f2e7a6f38a                      page_one_decay_risk|low_engagement_visible_page                   monitor           101078           2.7 0.85                   0
             4 content_e5ae436f9a16 page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page    refresh_and_review_ctr           117741           3.0 0.45                   0
             5 content_3430a8b94511 page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page    refresh_and_review_ctr           152617           3.3 0.29       

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.